# Module 31 Extension — Graph-Aware Loop Engineering

BUILD → TRY → BREAK → DEBUG → MEASURE. Vendor-neutral and deterministic.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Edge:
    source:str; relation:str; target:str; tenant:str; claim_id:str

edges=[Edge('A','OWNS','B','t1','c1'), Edge('B','SUPPORTS','C','t1','c2'), Edge('X','OWNS','Y','t2','c3')]

def bounded(start, tenant, hops=1):
    assert 0 <= hops <= 3
    seen={start}; frontier={start}
    for _ in range(hops):
        nxt=set()
        for e in edges:
            if e.tenant==tenant and e.source in frontier and e.target not in seen: nxt.add(e.target)
            if e.tenant==tenant and e.target in frontier and e.source not in seen: nxt.add(e.source)
        seen |= nxt; frontier=nxt
    return sorted(seen-{start})

print(bounded('A','t1',2))

## BREAK
Try `bounded('A','t1',99)`. The hard traversal budget must reject it. Then add a poisoned cross-tenant edge and verify it is excluded.

In [ ]:
try:
    bounded('A','t1',99)
except AssertionError as exc:
    print('Expected bounded-traversal failure:', exc)

assert bounded('A','t1',2)==['B','C']
assert bounded('A','t2',2)==[]
print('Graph observation security checks passed')

## MEASURE
Extend the loop benchmark with graph hops = 0/1/2/3 and record task success, traversal work, latency and token cost. Defend the point where extra graph context stops paying for itself.